# Simple RNN

## Load data

Set directory

In [2]:
import sys
import os

# Find the project root (Speciale_Kode)
current_dir = os.getcwd()
project_root = current_dir

# Looks for "Speciale_Kode" folder:
while os.path.basename(project_root) != "Speciale_Kode":
    project_root = os.path.dirname(project_root)

# Add to Python path
if project_root not in sys.path:
    sys.path.append(project_root)

Uses full feature set from read_data to predict DKPrice with cross-validation.

In [3]:
import pandas as pd

from Modules.read_data import read_data

PRICE_ZONE = "DK1"  # "DK1" or "DK2"
TRAIN_HOURS = 3 * 8760

(
    DK1_train,
    DK1_test,
    DK2_train,
    DK2_test,
    DK1_train_weather,
    DK1_test_weather,
    DK2_train_weather,
    DK2_test_weather
) = read_data("combined_data_cleaned_v5.csv")

if PRICE_ZONE == "DK1":
    dataset_train = DK1_train.copy()
    dataset_test = DK1_test.copy()
elif PRICE_ZONE == "DK2":
    dataset_train = DK2_train.copy()
    dataset_test = DK2_test.copy()
else:
    raise ValueError("PRICE_ZONE must be 'DK1' or 'DK2'.")

# Keep legacy variable names used by later cells in this notebook.
df = pd.concat([dataset_train, dataset_test], ignore_index=True).sort_values("Time").reset_index(drop=True)
target_time = pd.Timestamp("2024-01-01 00:00:00")
history = dataset_train.loc[dataset_train["Time"] < target_time].copy().sort_values("Time").reset_index(drop=True)
history = history.iloc[-TRAIN_HOURS:].copy().reset_index(drop=True)
price_series = history["DKPrice"].astype(float).reset_index(drop=True)
prices = price_series.to_numpy().reshape(-1, 1)
full_price_series = df["DKPrice"].astype(float).reset_index(drop=True)

print(f"Using zone: {PRICE_ZONE}")
print(f"Train history rows: {len(history)}")
print(f"Test shape: {dataset_test.shape}")
print("Model target: DKPrice only")

Notebook_dir: c:\Users\n_and\OneDrive\Delt skrivebord\Data Science\Speciale\Energinet\Delte scripts\Speciale_Kode\Modules
Python_dir: c:\Users\n_and\OneDrive\Delt skrivebord\Data Science\Speciale\Energinet\Delte scripts\Speciale_Kode
Data_folder: c:\Users\n_and\OneDrive\Delt skrivebord\Data Science\Speciale\Energinet\Delte scripts\Speciale_Kode\Data
Training data shape (DK1): (78888, 38)
Test data shape (DK1): (8760, 38)
Test set fraction (DK1): 9.99%
Training data shape (DK2): (78888, 38)
Test data shape (DK2): (8760, 38)
Test set fraction (DK2): 9.99%
Using zone: DK1
Train history rows: 26280
Test shape: (8760, 38)
Model target: DKPrice only


Test CUDA

In [3]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("CUDA DIAGNOSTICS")
print("\nBasic Info:")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {device}")

if torch.cuda.is_available():
    print(f"\nGPU Info:")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"cuDNN Version: {torch.backends.cudnn.version()}")
    print(f"Device Count: {torch.cuda.device_count()}")

    test_tensor = torch.randn(100, 100).to(device)
    print(f"Tensor on CUDA: {test_tensor.is_cuda}")

else:
    print("\n  Running on CPU - no CUDA available")

CUDA DIAGNOSTICS

Basic Info:
CUDA available: True
Device: cuda

GPU Info:
GPU Name: NVIDIA GeForce RTX 5060 Ti
CUDA Version: 12.8
cuDNN Version: 91002
Device Count: 1
Tensor on CUDA: True


### Helper functions

In [4]:
import copy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import DataLoader, TensorDataset


def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed: int) -> None:
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def smape_mean(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=np.float32).reshape(-1)
    y_pred = np.asarray(y_pred, dtype=np.float32).reshape(-1)
    denom = np.abs(y_true) + np.abs(y_pred)
    values = np.where(denom == 0, 0.0, 200.0 * np.abs(y_pred - y_true) / denom)
    return float(np.mean(values))


def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=np.float32).reshape(-1)
    y_pred = np.asarray(y_pred, dtype=np.float32).reshape(-1)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def build_supervised_sequences(values: np.ndarray, sequence_length: int):
    values = np.asarray(values, dtype=np.float32).reshape(-1, 1)
    if len(values) <= sequence_length:
        raise ValueError(
            f"Need more than sequence_length={sequence_length} points, got {len(values)}."
        )

    X = []
    y = []
    for index in range(len(values) - sequence_length):
        X.append(values[index : index + sequence_length])
        y.append(values[index + sequence_length, 0])

    return np.asarray(X, dtype=np.float32), np.asarray(y, dtype=np.float32)


def build_validation_folds(
    val_start: str,
    val_window: int,
    predict_period: int,
    stride: int,
):
    val_start_ts = pd.Timestamp(val_start)
    val_end_ts = val_start_ts + pd.Timedelta(hours=val_window)

    folds = []
    fold_start = val_start_ts
    fold_no = 1
    while fold_start + pd.Timedelta(hours=predict_period) <= val_end_ts:
        fold_end = fold_start + pd.Timedelta(hours=predict_period)
        folds.append(
            {
                "fold": fold_no,
                "val_start": fold_start,
                "val_end": fold_end,
            }
        )
        fold_no += 1
        fold_start = fold_start + pd.Timedelta(hours=stride)

    if not folds:
        raise ValueError("No validation folds could be created with the given settings.")

    return folds, val_start_ts, val_end_ts


def prepare_univariate_split(
    data: pd.DataFrame,
    train_window: int,
    val_window: int,
    val_start: str,
    predict_period: int,
    stride: int,
    include_remaining_2024: bool,
):
    data = data.copy().sort_values("Time").reset_index(drop=True)
    folds, val_start_ts, val_end_ts = build_validation_folds(
        val_start=val_start,
        val_window=val_window,
        predict_period=predict_period,
        stride=stride,
    )

    train_end_ts = val_start_ts
    train_start_ts = train_end_ts - pd.Timedelta(hours=train_window)

    data_min = data["Time"].min()
    if train_start_ts < data_min:
        raise ValueError(
            f"Not enough history for train_window={train_window}. "
            f"Need data from {train_start_ts}, but dataset starts at {data_min}."
        )

    base_train = data.loc[
        (data["Time"] >= train_start_ts) & (data["Time"] < train_end_ts),
        ["Time", "DKPrice"],
    ].copy()

    val_frame = data.loc[
        (data["Time"] >= val_start_ts) & (data["Time"] < val_end_ts),
        ["Time", "DKPrice"],
    ].copy()

    if include_remaining_2024:
        remaining_2024 = val_frame.copy()
        remaining_2024["is_validation"] = False
        for fold in folds:
            mask = (remaining_2024["Time"] >= fold["val_start"]) & (remaining_2024["Time"] < fold["val_end"])
            remaining_2024.loc[mask, "is_validation"] = True

        extra_train = remaining_2024.loc[~remaining_2024["is_validation"]].drop(columns=["is_validation"])
        train_frame = pd.concat([base_train, extra_train], ignore_index=True).sort_values("Time").reset_index(drop=True)
    else:
        train_frame = base_train.sort_values("Time").reset_index(drop=True)

    return train_frame, val_frame, folds


class UnivariateSimpleRNN(nn.Module):
    def __init__(self, hidden_size: int, layers: int, dropout: float = 0.0):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=1,
            hidden_size=hidden_size,
            num_layers=layers,
            dropout=float(dropout) if layers > 1 else 0.0,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        output, _ = self.rnn(x)
        return self.fc(output[:, -1, :])


class TorchRNNRegressor(BaseEstimator, RegressorMixin):
    """GPU-aware univariate RNN regressor with warm-start support."""

    def __init__(
        self,
        hidden_size: int = 32,
        layers: int = 1,
        learning_rate: float = 1e-3,
        epochs: int = 1,
        batch_size: int = 64,
        random_state: int = 42,
        sequence_length: int = 24,
        dropout: float = 0.0,
        warm_start: bool = False,
    ):
        self.hidden_size = hidden_size
        self.layers = layers
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        self.random_state = random_state
        self.sequence_length = sequence_length
        self.dropout = dropout
        self.warm_start = warm_start
        self.device_ = None
        self.model_ = None
        self.optimizer_ = None
        self.loss_fn_ = None

    def _get_device(self):
        if self.device_ is None:
            self.device_ = get_device()
        return self.device_

    def _prepare_X(self, X):
        X_np = np.asarray(X, dtype=np.float32)
        if X_np.ndim == 2:
            X_np = X_np[:, :, None]
        if X_np.ndim != 3:
            raise ValueError(f"Expected X with shape (n_samples, sequence_length, 1), got {X_np.shape}.")
        return X_np

    def fit(self, X, y):
        device = self._get_device()
        X_np = self._prepare_X(X)
        y_np = np.asarray(y, dtype=np.float32).reshape(-1, 1)
        X_tensor = torch.tensor(X_np, dtype=torch.float32)
        y_tensor = torch.tensor(y_np, dtype=torch.float32)

        if self.model_ is None or not self.warm_start:
            set_seed(self.random_state)
            self.model_ = UnivariateSimpleRNN(
                hidden_size=int(self.hidden_size),
                layers=int(self.layers),
                dropout=float(self.dropout),
            ).to(device)
            self.optimizer_ = torch.optim.Adam(self.model_.parameters(), lr=float(self.learning_rate))
            self.loss_fn_ = nn.MSELoss()

        loader = DataLoader(TensorDataset(X_tensor, y_tensor), batch_size=int(self.batch_size), shuffle=True)

        self.model_.train()
        for _ in range(int(self.epochs)):
            for X_batch, y_batch in loader:
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)
                self.optimizer_.zero_grad()
                predictions = self.model_(X_batch)
                loss = self.loss_fn_(predictions, y_batch)
                loss.backward()
                self.optimizer_.step()

        return self

    def predict(self, X):
        device = self._get_device()
        X_np = self._prepare_X(X)
        X_tensor = torch.tensor(X_np, dtype=torch.float32).to(device)
        self.model_.eval()
        with torch.no_grad():
            predictions = self.model_(X_tensor).squeeze(-1).detach().cpu().numpy()
        return predictions


def recursive_forecast(model, history_values, horizon, sequence_length, scaler):
    history_values = np.asarray(history_values, dtype=np.float32).reshape(-1, 1)
    if len(history_values) < sequence_length:
        raise ValueError(
            f"Need at least sequence_length={sequence_length} history points, got {len(history_values)}."
        )

    scaled_history = scaler.transform(history_values)
    rolling_window = scaled_history[-sequence_length:].copy()
    predictions_scaled = []

    for _ in range(int(horizon)):
        next_scaled = float(model.predict(rolling_window[None, :, :])[0])
        predictions_scaled.append(next_scaled)
        rolling_window = np.vstack([rolling_window[1:], [[next_scaled]]])

    predictions = scaler.inverse_transform(np.asarray(predictions_scaled).reshape(-1, 1)).reshape(-1)
    return predictions


def train_with_validation(
    full_data: pd.DataFrame,
    train_frame: pd.DataFrame,
    val_frame: pd.DataFrame,
    folds: list,
    *,
    hidden_size: int,
    layers: int,
    learning_rate: float,
    batch_size: int,
    sequence_length: int,
    max_epochs: int,
    patience: int,
    random_state: int = 42,
    dropout: float = 0.0,
    wandb_run=None,
    log_prefix: str = "",
):
    device = get_device()
    scaler = MinMaxScaler()

    train_values = train_frame["DKPrice"].astype(float).to_numpy().reshape(-1, 1)
    train_scaled = scaler.fit_transform(train_values)
    X_train, y_train = build_supervised_sequences(train_scaled, sequence_length)

    model = TorchRNNRegressor(
        hidden_size=hidden_size,
        layers=layers,
        learning_rate=learning_rate,
        epochs=1,
        batch_size=batch_size,
        random_state=random_state,
        sequence_length=sequence_length,
        dropout=dropout,
        warm_start=True,
    )

    best_val_smape = float("inf")
    best_epoch = 0
    patience_counter = 0
    best_state_dict = None
    epoch_history = []
    fold_metric_rows = []
    start_time = pd.Timestamp.now(tz="UTC")

    for epoch in range(1, int(max_epochs) + 1):
        model.fit(X_train, y_train)

        train_pred_scaled = model.predict(X_train)
        train_pred = scaler.inverse_transform(train_pred_scaled.reshape(-1, 1)).reshape(-1)
        train_true = scaler.inverse_transform(y_train.reshape(-1, 1)).reshape(-1)
        train_smape = smape_mean(train_true, train_pred)
        train_rmse = rmse(train_true, train_pred)

        fold_true_values = []
        fold_pred_values = []
        current_fold_rows = []

        for fold in folds:
            history_values = full_data.loc[
                full_data["Time"] < fold["val_start"],
                "DKPrice",
            ].astype(float).to_numpy()
            fold_actual = full_data.loc[
                (full_data["Time"] >= fold["val_start"]) & (full_data["Time"] < fold["val_end"]),
                "DKPrice",
            ].astype(float).to_numpy()

            fold_pred = recursive_forecast(
                model=model,
                history_values=history_values,
                horizon=len(fold_actual),
                sequence_length=sequence_length,
                scaler=scaler,
            )

            fold_smape = smape_mean(fold_actual, fold_pred)
            fold_rmse = rmse(fold_actual, fold_pred)
            fold_mae = float(np.mean(np.abs(fold_actual - fold_pred)))

            fold_true_values.append(fold_actual)
            fold_pred_values.append(fold_pred)
            current_fold_rows.append(
                {
                    "epoch": epoch,
                    "fold": int(fold["fold"]),
                    "val_start": fold["val_start"],
                    "val_end": fold["val_end"],
                    "fold_smape": fold_smape,
                    "fold_rmse": fold_rmse,
                    "fold_mae": fold_mae,
                }
            )

        val_true = np.concatenate(fold_true_values)
        val_pred = np.concatenate(fold_pred_values)
        val_smape = smape_mean(val_true, val_pred)
        val_rmse = rmse(val_true, val_pred)

        improved = val_smape < best_val_smape
        if improved:
            best_val_smape = val_smape
            best_epoch = epoch
            patience_counter = 0
            best_state_dict = copy.deepcopy(model.model_.state_dict())
        else:
            patience_counter += 1

        elapsed_minutes = (pd.Timestamp.now(tz="UTC") - start_time).total_seconds() / 60.0
        epoch_row = {
            "epoch": epoch,
            "train_smape": train_smape,
            "val_smape": val_smape,
            "train_rmse": train_rmse,
            "val_rmse": val_rmse,
            "best_epoch": best_epoch,
            "best_val_smape": best_val_smape,
            "patience_counter": patience_counter,
            "elapsed_time_min": elapsed_minutes,
        }
        epoch_history.append(epoch_row)
        fold_metric_rows.extend(current_fold_rows)

        if wandb_run is not None:
            wandb_run.log({
                f"{log_prefix}epoch": epoch,
                f"{log_prefix}train_smape": train_smape,
                f"{log_prefix}val_smape": val_smape,
                f"{log_prefix}train_rmse": train_rmse,
                f"{log_prefix}val_rmse": val_rmse,
                f"{log_prefix}best_epoch": best_epoch,
                f"{log_prefix}best_val_smape": best_val_smape,
                f"{log_prefix}patience_counter": patience_counter,
                f"{log_prefix}elapsed_time_min": elapsed_minutes,
            })

        if patience_counter >= int(patience):
            break

    if best_state_dict is not None:
        model.model_.load_state_dict(best_state_dict)

    history_df = pd.DataFrame(epoch_history)
    fold_history_df = pd.DataFrame(fold_metric_rows)
    best_row = history_df.loc[history_df["epoch"] == best_epoch].iloc[0].to_dict()

    return {
        "model": model,
        "scaler": scaler,
        "history": history_df,
        "fold_history": fold_history_df,
        "best_epoch": int(best_epoch),
        "best_val_smape": float(best_val_smape),
        "best_train_smape": float(best_row["train_smape"]),
        "best_train_rmse": float(best_row["train_rmse"]),
        "best_val_rmse": float(best_row["val_rmse"]),
        "epochs_trained": int(len(history_df)),
        "device": device,
        "validation_rows": int(len(val_frame)),
    }

## Hyperparameter search

Search grid

In [5]:
import numpy as np

param_grid = {
    "hidden_size": [16, 32],
    "layers": [1, 2],
    "learning_rate": [0.001, 0.0005],
    "batch_size": [32, 64],
    "sequence_length": [24, 168],
    "dropout": [0.0, 0.2],
}

print("Total combinations:", np.prod([len(v) for v in param_grid.values()]))

Total combinations: 64


Hyperparameter search

In [6]:
import itertools
from pathlib import Path
from time import time

import pandas as pd
import wandb

TRAIN_WINDOW = 3 * 8760
VAL_WINDOW = 1 * 8784
VAL_START = "2024-01-01 00:00:00"
PREDICT_PERIOD = 4 * 168
STRIDE = 13 * 168
MAX_EPOCHS = 60
PATIENCE = 8
INCLUDE_REMAINING_2024 = True

num_combinations = np.prod([len(v) for v in param_grid.values()])
print(f"Total number of combinations to test: {num_combinations}")

param_names = list(param_grid.keys())
param_values = list(param_grid.values())
all_combinations = list(itertools.product(*param_values))

full_data = df.copy().sort_values("Time").reset_index(drop=True)
train_frame, val_frame, folds = prepare_univariate_split(
    data=full_data,
    train_window=TRAIN_WINDOW,
    val_window=VAL_WINDOW,
    val_start=VAL_START,
    predict_period=PREDICT_PERIOD,
    stride=STRIDE,
    include_remaining_2024=INCLUDE_REMAINING_2024,
)

WANDB_PROJECT = "RNN_uni_param_search_DK1"
WANDB_RUN_BASENAME = f"{PRICE_ZONE}_rnn_uni_param_search"

results = []
start_time = time()

for comb_number, combination in enumerate(all_combinations, start=1):
    params = dict(zip(param_names, combination))
    run_name = f"{WANDB_RUN_BASENAME}_comb_{comb_number:03d}"
    print(f"\nCombination {comb_number}/{num_combinations}: {params}")

    run = wandb.init(
        project=WANDB_PROJECT,
        name=run_name,
        config={
            "price_zone": PRICE_ZONE,
            "combination": int(comb_number),
            "num_combinations": int(num_combinations),
            "train_window": int(TRAIN_WINDOW),
            "val_window": int(VAL_WINDOW),
            "val_start": VAL_START,
            "predict_period": int(PREDICT_PERIOD),
            "stride": int(STRIDE),
            "max_epochs": int(MAX_EPOCHS),
            "patience": int(PATIENCE),
            "include_remaining_2024": bool(INCLUDE_REMAINING_2024),
            **params,
        },
        tags=["rnn", "hyperparameter-search", "univariate", "validation", "early-stopping"],
        reinit=True,
        settings=wandb.Settings(start_method="thread"),
    )

    try:
        result = train_with_validation(
            full_data=full_data,
            train_frame=train_frame,
            val_frame=val_frame,
            folds=folds,
            hidden_size=int(params["hidden_size"]),
            layers=int(params["layers"]),
            learning_rate=float(params["learning_rate"]),
            batch_size=int(params["batch_size"]),
            sequence_length=int(params["sequence_length"]),
            max_epochs=int(MAX_EPOCHS),
            patience=int(PATIENCE),
            random_state=42,
            dropout=float(params["dropout"]),
            wandb_run=run,
            log_prefix="combo_",
        )

        run.log({
            "epoch_history": wandb.Table(dataframe=result["history"]),
        })
        run.summary.update({
            "best_epoch": int(result["best_epoch"]),
            "best_val_smape": float(result["best_val_smape"]),
            "best_train_smape": float(result["best_train_smape"]),
            "best_train_rmse": float(result["best_train_rmse"]),
            "best_val_rmse": float(result["best_val_rmse"]),
            "epochs_trained": int(result["epochs_trained"]),
        })

        row = {
            **params,
            "price_zone": PRICE_ZONE,
            "train_window": int(TRAIN_WINDOW),
            "val_window": int(VAL_WINDOW),
            "val_start": VAL_START,
            "predict_period": int(PREDICT_PERIOD),
            "stride": int(STRIDE),
            "include_remaining_2024": bool(INCLUDE_REMAINING_2024),
            "best_epoch": int(result["best_epoch"]),
            "best_val_smape": float(result["best_val_smape"]),
            "best_train_smape": float(result["best_train_smape"]),
            "best_train_rmse": float(result["best_train_rmse"]),
            "best_val_rmse": float(result["best_val_rmse"]),
            "epochs_trained": int(result["epochs_trained"]),
        }
        results.append(row)

    finally:
        wandb.finish()

results_df = pd.DataFrame(results).sort_values("best_val_smape")

summary_run = wandb.init(
    project=WANDB_PROJECT,
    name=f"{WANDB_RUN_BASENAME}_summary",
    reinit=True,
    settings=wandb.Settings(start_method="thread"),
)
summary_run.log({
    "all_results": wandb.Table(dataframe=results_df),
    "best_hidden_size": int(results_df.iloc[0]["hidden_size"]),
    "best_layers": int(results_df.iloc[0]["layers"]),
    "best_learning_rate": float(results_df.iloc[0]["learning_rate"]),
    "best_batch_size": int(results_df.iloc[0]["batch_size"]),
    "best_sequence_length": int(results_df.iloc[0]["sequence_length"]),
    "best_dropout": float(results_df.iloc[0]["dropout"]),
    "best_epoch": int(results_df.iloc[0]["best_epoch"]),
    "best_val_smape": float(results_df.iloc[0]["best_val_smape"]),
    "total_time_min": (time() - start_time) / 60,
})
summary_run.summary.update({
    "best_epoch": int(results_df.iloc[0]["best_epoch"]),
    "best_val_smape": float(results_df.iloc[0]["best_val_smape"]),
    "total_time_min": (time() - start_time) / 60,
})
wandb.finish()

project_root = Path.cwd()
while project_root.name != "Speciale_Kode" and project_root.parent != project_root:
    project_root = project_root.parent

output_folder = project_root / "Deep learners" / "Simple RNN"
output_folder.mkdir(parents=True, exist_ok=True)

base_filename = f"{PRICE_ZONE}_rnn_hyperparameter_search_results"
filename = output_folder / f"{base_filename}.csv"
counter = 1
while filename.exists():
    filename = output_folder / f"{base_filename}_{counter}.csv"
    counter += 1

results_df.to_csv(filename, index=False, decimal=",")
print(f"\nResults saved to: {filename}")
print(f"Total time: {(time() - start_time) / 60:.2f} minutes")
display(results_df.head(10))

wandb: WARNING `start_method` is deprecated and will be removed in a future version of wandb. This setting is currently non-functional and safely ignored.


Total number of combinations to test: 64

Combination 1/64: {'hidden_size': 16, 'layers': 1, 'learning_rate': 0.001, 'batch_size': 32, 'sequence_length': 24, 'dropout': 0.0}


wandb: Currently logged in as: nande24 (Energinet_speciale) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


combo_best_epoch,▁▅█████████
combo_best_val_smape,█▂▁▁▁▁▁▁▁▁▁
combo_elapsed_time_min,▁▂▂▃▄▅▅▆▇▇█
combo_epoch,▁▂▂▃▄▅▅▆▇▇█
combo_patience_counter,▁▁▁▂▃▄▅▅▆▇█
combo_train_rmse,█▅▃▂▂▂▂▁▂▁▁
combo_train_smape,█▅▃▂▂▄▆▁█▂▂
combo_val_rmse,▁▂▁▁▄▄▃▃█▃▄
combo_val_smape,█▂▁█▅▅▄██▄▄
best_epoch,3
best_train_rmse,182.02493



Combination 2/64: {'hidden_size': 16, 'layers': 1, 'learning_rate': 0.001, 'batch_size': 32, 'sequence_length': 24, 'dropout': 0.2}


...


Combination 12/64: {'hidden_size': 16, 'layers': 1, 'learning_rate': 0.0005, 'batch_size': 32, 'sequence_length': 168, 'dropout': 0.2}


combo_best_epoch,▁▁▁▁▁▁▁█████████
combo_best_val_smape,███████▁▁▁▁▁▁▁▁▁
combo_elapsed_time_min,▁▁▂▂▃▃▄▄▅▅▆▆▇▇██
combo_epoch,▁▁▂▂▃▃▄▄▅▅▆▆▇▇██
combo_patience_counter,▁▂▃▄▅▅▆▁▂▃▄▅▅▆▇█
combo_train_rmse,█▇▅▄▃▂▂▂▂▂▁▂▁▁▁▁
combo_train_smape,██▆▄▃▃▂▁▂▄▂▄▂▂▁▁
combo_val_rmse,▄▅▄▄█▆▃▁█▅▅▅▅▅▅▄
combo_val_smape,▄▅▄▄█▅█▁▅▅▅██▅▄▄
best_epoch,8
best_train_rmse,152.66776



Combination 13/64: {'hidden_size': 16, 'layers': 1, 'learning_rate': 0.0005, 'batch_size': 64, 'sequence_length': 24, 'dropout': 0.0}


combo_best_epoch,▁▁▁███
combo_best_val_smape,███▁▁▁
combo_elapsed_time_min,▁▂▄▅▆█
combo_epoch,▁▂▄▅▇█
combo_patience_counter,▁▅█▁▅█
combo_train_rmse,█▃▂▂▁▁
combo_train_smape,█▃▃▂▁▂
combo_val_rmse,▂▃▃▁▂█
combo_val_smape,▂▄▄▁▅█
combo_best_epoch,4
combo_best_val_smape,53.61917


KeyboardInterrupt: 

## Train final model

In [ ]:
import pandas as pd
import wandb
import tempfile
import joblib
from pathlib import Path

# =========================
# TRAIN FINAL MODEL
# =========================
FINAL_HIDDEN_SIZE = 32
FINAL_LAYERS = 2
FINAL_LEARNING_RATE = 0.0005
FINAL_BATCH_SIZE = 32
FINAL_SEQUENCE_LENGTH = 48
FINAL_DROPOUT = 0.0
FINAL_MAX_EPOCHS = 80
FINAL_PATIENCE = 10
INCLUDE_REMAINING_2024 = True

TRAIN_WINDOW = 3 * 8760
VAL_WINDOW = 1 * 8784
VAL_START = "2024-01-01 00:00:00"
PREDICT_PERIOD = 4 * 168
STRIDE = 13 * 168

full_data = df.copy().sort_values("Time").reset_index(drop=True)
final_train_frame, final_val_frame, final_folds = prepare_univariate_split(
    data=full_data,
    train_window=TRAIN_WINDOW,
    val_window=VAL_WINDOW,
    val_start=VAL_START,
    predict_period=PREDICT_PERIOD,
    stride=STRIDE,
    include_remaining_2024=INCLUDE_REMAINING_2024,
)

# Ensure device is set
if 'device' not in globals():
    device = get_device()
    print(f"Device not previously set. Using: {device}")
else:
    print(f"Using existing device: {device}")

WANDB_PROJECT = "RNN_final_training"
WANDB_RUN_NAME = f"{PRICE_ZONE}_rnn_final_train"
wandb_final_run = wandb.init(
    project=WANDB_PROJECT,
    name=WANDB_RUN_NAME,
    config={
        "price_zone": PRICE_ZONE,
        "hidden_size": FINAL_HIDDEN_SIZE,
        "layers": FINAL_LAYERS,
        "learning_rate": FINAL_LEARNING_RATE,
        "batch_size": FINAL_BATCH_SIZE,
        "sequence_length": FINAL_SEQUENCE_LENGTH,
        "max_epochs": FINAL_MAX_EPOCHS,
        "patience": FINAL_PATIENCE,
        "train_window": int(TRAIN_WINDOW),
        "val_window": int(VAL_WINDOW),
        "val_start": VAL_START,
        "predict_period": int(PREDICT_PERIOD),
        "stride": int(STRIDE),
        "include_remaining_2024": bool(INCLUDE_REMAINING_2024),
        "device": str(device),
    },
    tags=["rnn", "final-training", "univariate", "validation", "early-stopping"],
    reinit=True,
    settings=wandb.Settings(start_method="thread"),
)

final_result = train_with_validation(
    full_data=full_data,
    train_frame=final_train_frame,
    val_frame=final_val_frame,
    folds=final_folds,
    hidden_size=FINAL_HIDDEN_SIZE,
    layers=FINAL_LAYERS,
    learning_rate=FINAL_LEARNING_RATE,
    batch_size=FINAL_BATCH_SIZE,
    sequence_length=FINAL_SEQUENCE_LENGTH,
    max_epochs=FINAL_MAX_EPOCHS,
    patience=FINAL_PATIENCE,
    random_state=42,
    dropout=FINAL_DROPOUT,
    wandb_run=wandb_final_run,
    log_prefix="final_",
)

model = final_result["model"].model_
scaler = final_result["scaler"]
SEQ_LEN = FINAL_SEQUENCE_LENGTH
EPOCHS = int(final_result["epochs_trained"])
LEARNING_RATE = FINAL_LEARNING_RATE
BEST_EPOCH = final_result["best_epoch"]
BEST_VAL_SMAPE = final_result["best_val_smape"]
BEST_TRAIN_SMAPE = final_result["best_train_smape"]
BEST_TRAIN_RMSE = final_result["best_train_rmse"]
BEST_VAL_RMSE = final_result["best_val_rmse"]

wandb_final_run.log({
    "epoch_history": wandb.Table(dataframe=final_result["history"]),
    "fold_history": wandb.Table(dataframe=final_result["fold_history"]),
})
wandb_final_run.summary.update({
    "best_epoch": int(BEST_EPOCH),
    "best_val_smape": float(BEST_VAL_SMAPE),
    "best_train_smape": float(BEST_TRAIN_SMAPE),
    "best_train_rmse": float(BEST_TRAIN_RMSE),
    "best_val_rmse": float(BEST_VAL_RMSE),
    "epochs_trained": int(final_result["epochs_trained"]),
})

with tempfile.TemporaryDirectory() as tmpdir:
    tmpdir_path = Path(tmpdir)
    model_path = tmpdir_path / "final_rnn_state_dict.pt"
    scaler_path = tmpdir_path / "final_rnn_scaler.joblib"
    metadata_path = tmpdir_path / "final_rnn_metadata.txt"

    torch.save(model.state_dict(), model_path)
    joblib.dump(scaler, scaler_path)
    metadata_path.write_text(
        "\n".join([
            f"price_zone={PRICE_ZONE}",
            f"best_epoch={BEST_EPOCH}",
            f"best_val_smape={BEST_VAL_SMAPE}",
            f"best_train_smape={BEST_TRAIN_SMAPE}",
            f"best_train_rmse={BEST_TRAIN_RMSE}",
            f"best_val_rmse={BEST_VAL_RMSE}",
            f"sequence_length={SEQ_LEN}",
            f"hidden_size={FINAL_HIDDEN_SIZE}",
            f"layers={FINAL_LAYERS}",
            f"learning_rate={FINAL_LEARNING_RATE}",
            f"batch_size={FINAL_BATCH_SIZE}",
        ]),
        encoding="utf-8",
    )

    artifact = wandb.Artifact(
        name=f"{WANDB_RUN_NAME}_artifact",
        type="model",
        metadata={
            "price_zone": PRICE_ZONE,
            "best_epoch": int(BEST_EPOCH),
            "best_val_smape": float(BEST_VAL_SMAPE),
            "best_train_smape": float(BEST_TRAIN_SMAPE),
            "best_train_rmse": float(BEST_TRAIN_RMSE),
            "best_val_rmse": float(BEST_VAL_RMSE),
        },
    )
    artifact.add_file(str(model_path), name="model_state_dict.pt")
    artifact.add_file(str(scaler_path), name="scaler.joblib")
    artifact.add_file(str(metadata_path), name="metadata.txt")
    wandb_final_run.log_artifact(artifact)

print(f"Best epoch: {BEST_EPOCH}")
print(f"Best val SMAPE: {BEST_VAL_SMAPE:.3f}")
print(f"Best train SMAPE: {BEST_TRAIN_SMAPE:.3f}")
print(f"Best train RMSE: {BEST_TRAIN_RMSE:.4f}")
print(f"Best val RMSE: {BEST_VAL_RMSE:.4f}")

wandb.finish()

Epoch 1/20, Loss: 0.062588
Epoch 2/20, Loss: 0.054383
Epoch 3/20, Loss: 0.046841
Epoch 4/20, Loss: 0.039962
Epoch 5/20, Loss: 0.033742
Epoch 6/20, Loss: 0.028170
Epoch 7/20, Loss: 0.023234
Epoch 8/20, Loss: 0.018915
Epoch 9/20, Loss: 0.015189
Epoch 10/20, Loss: 0.012029
Epoch 11/20, Loss: 0.009401
Epoch 12/20, Loss: 0.007268
Epoch 13/20, Loss: 0.005588
Epoch 14/20, Loss: 0.004317
Epoch 15/20, Loss: 0.003407
Epoch 16/20, Loss: 0.002810
Epoch 17/20, Loss: 0.002478
Epoch 18/20, Loss: 0.002361
Epoch 19/20, Loss: 0.002413
Epoch 20/20, Loss: 0.002589

Price zone: DK1
Predicted time: 2024-01-01 00:00:00
Last 5 prices before prediction: [260.100006 220.660004 213.729996 200.309998 126.660004]
Predicted DKPrice: 783.51


Evaluate model

In [ ]:
# =========================
# EVALUATE MODEL IN 168-HOUR BLOCKS
# =========================
from pathlib import Path

EVAL_START_OFFSET_HOURS = 0   # fx 0 = start ved TARGET_TIME, 24 = start 1 dag efter
EVAL_BLOCK_HOURS = 168        # hver prognoseblok er 168 timer
EVAL_BLOCKS = 1               # ændr denne hvis du vil evaluere flere blokke
EVAL_HOURS = EVAL_BLOCK_HOURS * EVAL_BLOCKS

# Ensure device is available
if 'device' not in globals():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device not previously set. Using: {device}")
else:
    print(f"Using device: {device}")

eval_start_time = target_time + pd.Timedelta(hours=EVAL_START_OFFSET_HOURS)
eval_end_time = eval_start_time + pd.Timedelta(hours=EVAL_HOURS)

# Brug den fulde zoneserie fra df (som allerede er filtreret til PRICE_ZONE)
zone_df = df.sort_values("Time").reset_index(drop=True)

actual_eval = zone_df.loc[
    (zone_df["Time"] >= eval_start_time) & (zone_df["Time"] < eval_end_time),
    "DKPrice",
].astype(float).values

if len(actual_eval) < EVAL_HOURS:
    raise ValueError(
        f"Ikke nok fremtidige data fra {eval_start_time} til {eval_end_time}. "
        f"Har {len(actual_eval)} timer, men skal bruge {EVAL_HOURS}."
    )

actual_eval = actual_eval[:EVAL_HOURS]

def recursive_block_forecast(actual_series, series_before_start, block_hours, model, scaler, device):
    predictions = []
    block_smapes = []
    block_maes = []
    block_rmses = []
    day_smapes_by_position = [[] for _ in range(7)]
    daily_smapes_all = []

    for block_start in range(0, len(actual_series), block_hours):
        block_end = min(block_start + block_hours, len(actual_series))
        block_actual = actual_series[block_start:block_end]

        if block_start == 0:
            block_history = series_before_start[-SEQ_LEN:]
        else:
            block_history = actual_series[block_start - SEQ_LEN:block_start]

        if len(block_history) < SEQ_LEN:
            raise ValueError(
                f"Not enough true history to start block at hour {block_start}. "
                f"Need {SEQ_LEN}, got {len(block_history)}."
            )

        seq_scaled = scaler.transform(np.asarray(block_history).reshape(-1, 1))
        block_predictions = []

        model.eval()
        with torch.no_grad():
            for _ in range(len(block_actual)):
                x_in = torch.tensor(seq_scaled, dtype=torch.float32).unsqueeze(0).to(device)
                next_scaled = model(x_in).cpu().numpy()[0, 0]
                next_pred = scaler.inverse_transform(np.array([[next_scaled]]))[0, 0]
                block_predictions.append(next_pred)
                seq_scaled = np.vstack([seq_scaled[1:], [[next_scaled]]])

        block_predictions = np.array(block_predictions)
        predictions.extend(block_predictions.tolist())

        block_smape = smape_mean(block_actual, block_predictions)
        block_mae = float(np.mean(np.abs(block_actual - block_predictions)))
        block_rmse = float(np.sqrt(np.mean((block_actual - block_predictions) ** 2)))
        block_smapes.append(block_smape)
        block_maes.append(block_mae)
        block_rmses.append(block_rmse)

        for day_idx in range(7):
            day_start = day_idx * 24
            day_end = min(day_start + 24, len(block_actual))
            if day_start >= len(block_actual):
                break
            day_smape = smape_mean(block_actual[day_start:day_end], block_predictions[day_start:day_end])
            day_smapes_by_position[day_idx].append(day_smape)
            daily_smapes_all.append(day_smape)

    return np.array(predictions), block_smapes, block_maes, block_rmses, day_smapes_by_position, daily_smapes_all

pre_eval_series = zone_df.loc[zone_df["Time"] < eval_start_time, "DKPrice"].astype(float).values
if len(pre_eval_series) < SEQ_LEN:
    raise ValueError(
        f"Ikke nok historik før {eval_start_time}. "
        f"Har {len(pre_eval_series)} timer, skal bruge mindst {SEQ_LEN}."
    )

pred_eval, block_smapes, block_maes, block_rmses, day_smapes_by_position, daily_smapes_all = recursive_block_forecast(
    actual_series=actual_eval,
    series_before_start=pre_eval_series,
    block_hours=EVAL_BLOCK_HOURS,
    model=model,
    scaler=scaler,
    device=device,
)

avg_weekly_smape = float(np.mean(block_smapes)) if block_smapes else float("nan")
avg_mae = float(np.mean(block_maes)) if block_maes else float("nan")
avg_rmse = float(np.mean(block_rmses)) if block_rmses else float("nan")

avg_day_smapes = [
    float(np.mean(values)) if values else float("nan")
    for values in day_smapes_by_position
]

max_avg_daily_smape = float(np.max(daily_smapes_all)) if daily_smapes_all else float("nan")
min_avg_daily_smape = float(np.min(daily_smapes_all)) if daily_smapes_all else float("nan")

print("\n=========================")
print("BLOCKED RECURSIVE EVALUATION")
print("=========================")
print(f"Eval device: {device}")
print(f"Eval start: {eval_start_time}")
print(f"Eval end  : {eval_end_time}")
print(f"Hours     : {EVAL_HOURS}")
print(f"Block size: {EVAL_BLOCK_HOURS}")
print(f"Blocks    : {EVAL_BLOCKS}")
print(f"Avg SMAPE : {avg_weekly_smape:.2f}%")
print(f"Avg MAE   : {avg_mae:.4f}")
print(f"Avg RMSE  : {avg_rmse:.4f}")
print(f"Max daily SMAPE in window: {max_avg_daily_smape:.2f}%")
print(f"Min daily SMAPE in window: {min_avg_daily_smape:.2f}%")

for i, (smape_value, mae_value, rmse_value) in enumerate(zip(block_smapes, block_maes, block_rmses), start=1):
    print(f"Block {i}: SMAPE={smape_value:.2f}%, MAE={mae_value:.4f}, RMSE={rmse_value:.4f}")

for i, value in enumerate(avg_day_smapes, start=1):
    print(f"Avg SMAPE day {i}: {value:.2f}%")

# =========================
# SAVE RESULTS TO CSV
# =========================
train_start_time = history["Time"].min()
train_end_time = history["Time"].max()

results_row = {
    "price_zone": PRICE_ZONE,
    "seq_len": SEQ_LEN,
    "train_hours": TRAIN_HOURS,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "target_time": str(target_time),
    "train_start_time": str(train_start_time),
    "train_end_time": str(train_end_time),
    "validation_start_time": str(eval_start_time),
    "validation_end_time": str(eval_end_time),
    "validation_hours": EVAL_HOURS,
    "eval_start_offset_hours": EVAL_START_OFFSET_HOURS,
    "eval_block_hours": EVAL_BLOCK_HOURS,
    "eval_blocks": EVAL_BLOCKS,
    "device": str(device),
    "avg_weekly_smape": float(avg_weekly_smape),
    "avg_weekly_mae": float(avg_mae),
    "avg_weekly_rmse": float(avg_rmse),
    "avg_daily_smape": float(np.mean(daily_smapes_all)),
    "max_avg_daily_smape_pct": float(max_avg_daily_smape),
    "min_avg_daily_smape_pct": float(min_avg_daily_smape),
}

for i, value in enumerate(avg_day_smapes, start=1):
    results_row[f"smape_day_{i}_pct"] = float(value)

for i, value in enumerate(block_smapes, start=1):
    results_row[f"block_{i}_smape_pct"] = float(value)
for i, value in enumerate(block_maes, start=1):
    results_row[f"block_{i}_mae"] = float(value)
for i, value in enumerate(block_rmses, start=1):
    results_row[f"block_{i}_rmse"] = float(value)

results_df = pd.DataFrame([results_row])

notebook_dir = Path.cwd()
file_index = 1
while True:
    csv_name = f"rnn_{PRICE_ZONE}_eval_{file_index}.csv"
    csv_path = notebook_dir / csv_name
    if not csv_path.exists():
        break
    file_index += 1

results_df.to_csv(csv_path, index=False, sep=";", decimal=",")
print(f"\nSaved evaluation CSV: {csv_path}")


SMAPE EVALUATION
Eval start: 2024-01-01 00:00:00
Eval end  : 2024-01-08 00:00:00
Hours     : 168
Avg daily SMAPE : 54.15%
Avg weekly SMAPE: 54.15%
SMAPE day 1: 115.07%
SMAPE day 2: 78.33%
SMAPE day 3: 89.38%
SMAPE day 4: 33.44%
SMAPE day 5: 18.63%
SMAPE day 6: 21.11%
SMAPE day 7: 23.12%

Saved evaluation CSV: c:\Users\n_and\OneDrive\Delt skrivebord\Data Science\Speciale\Energinet\Delte scripts\Speciale_Kode\Deep learners\Simple RNN\rnn_DK1_eval_5.csv
